# YOLO (BIRD DETECTION)

### YOLO MODEL TRAINING

In [ ]:
from ultralytics import YOLO
import os

model = YOLO('yolov8s.pt')

# Path to your dataset configuration file
data_yaml_path = 'D:/Project_P1/data/data.yaml'

# Train the model with explicit hyperparameters
results = model.train(
    data=data_yaml_path,
    epochs=100,
    imgsz=640,
    # CRITICAL FIX: Manually set optimizer and learning rate
    optimizer='SGD',      # Use SGD explicitly
    lr0=0.001,           # Start with a much lower initial learning rate
    momentum=0.9,         # Standard momentum value
    # Additional recommended settings for stability
    warmup_epochs=3,      # Gradually ramp up LR at start
    patience=10,          # Stop if no improvement for 10 epochs
    # Optional: Monitor closely on first run
    batch=8,             # Ensure this matches your GPU memory capacity
    # Visualization
    plots=True            # Generate helpful plots during training
)

# The trained model is saved in the 'runs' directory
best_model_path = os.path.join(results.save_dir, 'weights', 'best.pt')

print(f"Training complete. The best model is saved at: {best_model_path}")

### TESTING AND DOCUMENTING RESULTS OF YOLO MODEL

#### Bird Detection from Photo or Videos

In [8]:
import os
import cv2
import json
import pandas as pd
from ultralytics import YOLO
import matplotlib.pyplot as plt
import glob

# Load the trained model
best_model_path = "D:/Project_P1/src/ml/runs/detect/train2/weights/best.pt"
model = YOLO(best_model_path)
print("Trained model loaded successfully. Let's start inference! 🚀")

# Function to process and save bounding box data
def process_and_save_bboxes(results, source_path, output_dir, source_type="image"):
    os.makedirs(output_dir, exist_ok=True)
    
    # For storing all bounding box data
    all_bbox_data = []
    
    if source_type == "image":
        # Process single image
        result = results[0]
        image_name = os.path.basename(source_path)
        
        # Save the image with bounding boxes
        output_image_path = os.path.join(output_dir, f"{os.path.splitext(image_name)[0]}.jpg")
        image_with_boxes = result.plot()  # Draw bounding boxes on the image
        cv2.imwrite(output_image_path, image_with_boxes)

        
        # Extract and save bounding box data
        bbox_data = []
        if result.boxes is not None:
            for box in result.boxes:
                xyxy = box.xyxy[0].cpu().numpy()  # Get coordinates in xyxy format
                conf = box.conf[0].cpu().numpy()   # Get confidence score
                cls = box.cls[0].cpu().numpy()     # Get class ID
                
                bbox_info = {
                    "image": image_name,
                    "x1": float(xyxy[0]),
                    "y1": float(xyxy[1]),
                    "x2": float(xyxy[2]),
                    "y2": float(xyxy[3]),
                    "confidence": float(conf),
                    "class": int(cls),
                    "frame": 0  # For images, we use frame 0
                }
                bbox_data.append(bbox_info)
                all_bbox_data.append(bbox_info)
        
        # Save bounding box data to JSON
        json_path = os.path.join(output_dir, f"{os.path.splitext(image_name)[0]}_bboxes.json")
        with open(json_path, 'w') as f:
            json.dump(bbox_data, f, indent=4)
            
    elif source_type == "video":
        # Process video frame by frame
        cap = cv2.VideoCapture(source_path)
        frame_count = 0
        
        # Create video writer for output
        frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        
        video_name = os.path.basename(source_path)
        output_video_path = os.path.join(output_dir, video_name)
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))
        
        # Process each frame
        for i, result in enumerate(results):
            # Get the frame with bounding boxes drawn
            frame_with_boxes = result.plot()
            
            # Write frame to output video
            out.write(frame_with_boxes)
            
            # Extract and save bounding box data for this frame
            if result.boxes is not None:
                for box in result.boxes:
                    xyxy = box.xyxy[0].cpu().numpy()
                    conf = box.conf[0].cpu().numpy()
                    cls = box.cls[0].cpu().numpy()
                    
                    bbox_info = {
                        "video": video_name,
                        "x1": float(xyxy[0]),
                        "y1": float(xyxy[1]),
                        "x2": float(xyxy[2]),
                        "y2": float(xyxy[3]),
                        "confidence": float(conf),
                        "class": int(cls),
                        "frame": frame_count
                    }
                    all_bbox_data.append(bbox_info)
            
            frame_count += 1
        
        # Release video objects
        cap.release()
        out.release()
        
        # Save all bounding box data to JSON
        json_path = os.path.join(output_dir, f"{os.path.splitext(video_name)[0]}_bboxes.json")
        with open(json_path, 'w') as f:
            json.dump(all_bbox_data, f, indent=4)
    
    return all_bbox_data

# Function to visualize bird movement
def visualize_bird_movement(bbox_data, output_path):
    # Convert to DataFrame for easier manipulation
    df = pd.DataFrame(bbox_data)
    
    if len(df) == 0:
        print("No bird detections found for movement analysis.")
        return
    
    # Calculate center points of bounding boxes
    df['center_x'] = (df['x1'] + df['x2']) / 2
    df['center_y'] = (df['y1'] + df['y2']) / 2
    
    # Group by frame and find the primary bird (highest confidence)
    primary_birds = df.loc[df.groupby('frame')['confidence'].idxmax()]
    
    # Create visualization
    plt.figure(figsize=(10, 8))
    
    # Plot the movement path
    plt.plot(primary_birds['center_x'], primary_birds['center_y'], 'b-', alpha=0.6, label='Movement Path')
    plt.scatter(primary_birds['center_x'], primary_birds['center_y'], c=primary_birds['frame'], 
                cmap='viridis', s=50, alpha=0.8)
    
    # Add direction arrows
    for i in range(0, len(primary_birds)-1, max(1, len(primary_birds)//10)):
        dx = primary_birds['center_x'].iloc[i+1] - primary_birds['center_x'].iloc[i]
        dy = primary_birds['center_y'].iloc[i+1] - primary_birds['center_y'].iloc[i]
        plt.arrow(primary_birds['center_x'].iloc[i], primary_birds['center_y'].iloc[i], 
                 dx, dy, shape='full', color='red', length_includes_head=True, head_width=10)
    
    plt.colorbar(label='Frame Number')
    plt.title('Bird Movement Analysis')
    plt.xlabel('X Coordinate')
    plt.ylabel('Y Coordinate')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Save the visualization
    # plt.savefig(output_path, dpi=300, bbox_inches='tight')
    # plt.close()
    
    print(f"Movement visualization saved to: {output_path}")

# Test with a single image
test_data_dir = "C:/Users/visha/OneDrive/Desktop"
single_image_path = os.path.join(test_data_dir, "20250910_143325.heic")
print("\n--- Running inference on a single image ---")

results_single_image = model.predict(
    source=single_image_path,
    save=False,  # We'll handle saving ourselves to get the bbox data
    conf=0.25 
)

# Process and save bounding boxes
image_output_dir = "D:/Project_P1/results/image_analysis"
image_bbox_data = process_and_save_bboxes(results_single_image, single_image_path, image_output_dir, "image")
print(f"Inference on single image complete. Results saved at: {image_output_dir}")


# test_data_dir = "D:/Project_P1/data/valid"
# images_source_dir = os.path.join(test_data_dir, "images")

# # 2. Define where to save the output
# output_dir = "D:/Project_P1/results/image_analysis"

# # 3. Create the output directory if it doesn't exist to prevent errors
# os.makedirs(output_dir, exist_ok=True)

# # --- Find all image files ---
# # Create a list of all files with common image extensions (.jpg, .png, etc.)
# # This is more robust than just listing all files in the directory.
# image_extensions = ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.gif"]
# image_paths = []
# for ext in image_extensions:
#     image_paths.extend(glob.glob(os.path.join(images_source_dir, ext)))

# print(f"--- Found {len(image_paths)} images to process in '{images_source_dir}' ---")

# # --- Loop through each image and run inference ---
# all_bbox_data = [] # Optional: A list to store results from all images

# for image_path in image_paths:
#     print(f"\nProcessing: {os.path.basename(image_path)}...")
    
#     # Run prediction on the current image
#     results = model.predict(
#         source=image_path,
#         save=False,  # We'll handle saving ourselves
#         conf=0.25
#     )
    
#     # Process and save bounding boxes for the current image
#     # Your existing function should work perfectly here
#     image_bbox_data = process_and_save_bboxes(results, image_path, output_dir, "image")
    
#     # Optional: Append the results to our master list
#     if image_bbox_data:
#         all_bbox_data.append(image_bbox_data)

# print(f"\n--- Inference on all images complete. Results saved in: {output_dir} ---")


# # Test with a video
# video_file_path = "D:/Project_P1/data/valid/test_video.mp4"
# print("\n--- Running inference on a video file ---")

# results_video = model.predict(
#     source=video_file_path,
#     save=False,  # We'll handle saving ourselves to get the bbox data
#     conf=0.25
# )

# # Process and save bounding boxes
# video_output_dir = "D:/Project_P1/results/video_analysis"
# video_bbox_data = process_and_save_bboxes(results_video, video_file_path, video_output_dir, "video")
# print(f"Inference on video complete. Results saved at: {video_output_dir}")

# # Visualize bird movement from the video
# movement_visualization_path = os.path.join(video_output_dir, "bird_movement_analysis.png")
# visualize_bird_movement(video_bbox_data, movement_visualization_path)

# # Optional: Create a CSV file for further analysis
# csv_path = os.path.join(video_output_dir, "bird_detections.csv")
# pd.DataFrame(video_bbox_data).to_csv(csv_path, index=False)
# print(f"Detailed detection data saved to: {csv_path}")

# print("\nAnalysis complete! You can now examine:")
# print("1. Annotated images/videos with bounding boxes")
# print("2. JSON files with detailed detection data")
# print("3. Movement visualization plot")
# print("4. CSV file for further analysis")

Trained model loaded successfully. Let's start inference! 🚀

--- Running inference on a single image ---

image 1/1 C:\Users\visha\OneDrive\Desktop\20250910_143325.heic: 640x288 (no detections), 129.3ms
Speed: 6.0ms preprocess, 129.3ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 288)
Inference on single image complete. Results saved at: D:/Project_P1/results/image_analysis


#### Bird Movement Analysis

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from datetime import datetime

# Access the stored detection data
video_output_dir = "D:/Project_P1/results/video_analysis"
csv_path = os.path.join(video_output_dir, "bird_detections.csv")
# json_path = os.path.join(video_output_dir, f"{os.path.splitext(os.path.basename(video_file_path))[0]}_bboxes.json")

# Load the detection data
print("Loading detection data...")
# Try loading from CSV first
df_detections = pd.read_csv(csv_path)
print(f"Loaded {len(df_detections)} detections from CSV")

# Display basic info about the data
print("\n=== DETECTION DATA INFO ===")
print(f"Total frames with detections: {df_detections['frame'].nunique()}")
print(f"Total bird detections: {len(df_detections)}")
print(f"Date range: {df_detections['frame'].min()} to {df_detections['frame'].max()} frames")

# Calculate center points
df_detections['center_x'] = (df_detections['x1'] + df_detections['x2']) / 2
df_detections['center_y'] = (df_detections['y1'] + df_detections['y2']) / 2

# Group into discrete sequences based on frame gaps
def identify_sequences(df, frame_gap_threshold=10):
    """Identify discrete sequences based on gaps between frames"""
    # Sort by frame
    df_sorted = df.sort_values('frame')
    
    # Calculate frame differences
    frame_diffs = df_sorted['frame'].diff()
    
    # Identify sequence breaks where gap is larger than threshold
    sequence_breaks = (frame_diffs > frame_gap_threshold).cumsum()
    df_sorted['sequence_id'] = sequence_breaks
    
    return df_sorted

# Identify sequences
df_with_sequences = identify_sequences(df_detections, frame_gap_threshold=15)

# Analyze each sequence
sequences_info = []
for seq_id, seq_data in df_with_sequences.groupby('sequence_id'):
    frame_min = seq_data['frame'].min()
    frame_max = seq_data['frame'].max()
    frame_count = seq_data['frame'].nunique()
    bird_count = len(seq_data)
    
    sequences_info.append({
        'sequence_id': seq_id,
        'start_frame': frame_min,
        'end_frame': frame_max,
        'frame_count': frame_count,
        'bird_count': bird_count,
        'avg_birds_per_frame': round(bird_count / frame_count, 2) if frame_count > 0 else 0
    })

# Create sequences dataframe
df_sequences = pd.DataFrame(sequences_info)
print("\n=== DISCRETE SEQUENCES IDENTIFIED ===")
print(df_sequences.to_string(index=False))

# Function to test a specific sequence
def test_sequence(sequence_id, df_detections):
    """Test and analyze a specific sequence"""
    seq_data = df_detections[df_detections['sequence_id'] == sequence_id].copy()
    
    if len(seq_data) == 0:
        print(f"No data found for sequence {sequence_id}")
        return
    
    print(f"\n=== TESTING SEQUENCE {sequence_id} ===")
    print(f"Frames: {seq_data['frame'].min()} to {seq_data['frame'].max()}")
    print(f"Total detections: {len(seq_data)}")
    
    # Calculate movement metrics
    seq_data.sort_values('frame', inplace=True)
    
    # Calculate velocity between frames
    seq_data['dx'] = seq_data['center_x'].diff()
    seq_data['dy'] = seq_data['center_y'].diff()
    seq_data['velocity'] = np.sqrt(seq_data['dx']**2 + seq_data['dy']**2)
    
    # Filter out extreme values
    seq_data = seq_data[seq_data['velocity'] < 100]  # Adjust threshold as needed
    
    if len(seq_data) > 1:
        avg_velocity = seq_data['velocity'].mean()
        max_velocity = seq_data['velocity'].max()
        
        print(f"Average velocity: {avg_velocity:.2f} pixels/frame")
        print(f"Maximum velocity: {max_velocity:.2f} pixels/frame")
        
        # Determine general direction
        x_start, x_end = seq_data['center_x'].iloc[0], seq_data['center_x'].iloc[-1]
        y_start, y_end = seq_data['center_y'].iloc[0], seq_data['center_y'].iloc[-1]
        
        if abs(x_end - x_start) > abs(y_end - y_start):
            direction = "right" if x_end > x_start else "left"
        else:
            direction = "down" if y_end > y_start else "up"
            
        print(f"Overall movement direction: {direction}")
        
        return {
            'sequence_id': sequence_id,
            'avg_velocity': avg_velocity,
            'max_velocity': max_velocity,
            'direction': direction,
            'detection_count': len(seq_data)
        }
    
    return None

# Test all sequences
print("\n=== TESTING ALL SEQUENCES ===")
sequence_results = []
for seq_id in df_sequences['sequence_id'].unique():
    result = test_sequence(seq_id, df_with_sequences)
    if result:
        sequence_results.append(result)

# Create results dataframe
if sequence_results:
    df_results = pd.DataFrame(sequence_results)
    print("\n=== TEST RESULTS SUMMARY ===")
    print(df_results.to_string(index=False))
    
    # Save results to CSV
    results_path = os.path.join(video_output_dir, "sequence_test_results.csv")
    df_results.to_csv(results_path, index=False)
    print(f"\nResults saved to: {results_path}")
else:
    print("No valid sequences found for testing.")

# Advanced analysis: Detect potential flight patterns
def analyze_flight_patterns(df_detections):
    """Analyze flight patterns across sequences"""
    patterns = []
    
    for seq_id, seq_data in df_detections.groupby('sequence_id'):
        if len(seq_data) < 5:  # Need enough points for pattern analysis
            continue
            
        seq_data = seq_data.sort_values('frame')
        
        # Calculate derivatives (approximate velocity and acceleration)
        x = seq_data['center_x'].values
        y = seq_data['center_y'].values
        frames = seq_data['frame'].values
        
        # Calculate velocities
        dx = np.gradient(x, frames)
        dy = np.gradient(y, frames)
        velocity = np.sqrt(dx**2 + dy**2)
        
        # Calculate accelerations
        ddx = np.gradient(dx, frames)
        ddy = np.gradient(dy, frames)
        acceleration = np.sqrt(ddx**2 + ddy**2)
        
        # Classify pattern based on statistics
        avg_vel = np.mean(velocity)
        std_vel = np.std(velocity)
        avg_acc = np.mean(acceleration)
        
        if std_vel < 5 and avg_acc < 2:
            pattern_type = "Steady flight"
        elif avg_acc > 5:
            pattern_type = "Maneuvering"
        elif std_vel > 10:
            pattern_type = "Erratic flight"
        else:
            pattern_type = "Unknown"
        
        patterns.append({
            'sequence_id': seq_id,
            'pattern_type': pattern_type,
            'avg_velocity': avg_vel,
            'velocity_std': std_vel,
            'avg_acceleration': avg_acc,
            'detection_count': len(seq_data)
        })
    
    return pd.DataFrame(patterns)

# Analyze flight patterns
print("\n=== ANALYZING FLIGHT PATTERNS ===")
df_patterns = analyze_flight_patterns(df_with_sequences)
if not df_patterns.empty:
    print(df_patterns.to_string(index=False))
    
    # Save patterns to CSV
    patterns_path = os.path.join(video_output_dir, "flight_patterns.csv")
    df_patterns.to_csv(patterns_path, index=False)
    print(f"Flight patterns saved to: {patterns_path}")
else:
    print("Not enough data for pattern analysis.")

print("\n=== TESTING COMPLETE ===")
print("All sequences have been processed and analyzed.")

Loading detection data...
Loaded 390 detections from CSV

=== DETECTION DATA INFO ===
Total frames with detections: 390
Total bird detections: 390
Date range: 0 to 389 frames

=== DISCRETE SEQUENCES IDENTIFIED ===
 sequence_id  start_frame  end_frame  frame_count  bird_count  avg_birds_per_frame
           0            0        389          390         390                  1.0

=== TESTING ALL SEQUENCES ===

=== TESTING SEQUENCE 0 ===
Frames: 0 to 389
Total detections: 390
Average velocity: 1.49 pixels/frame
Maximum velocity: 5.91 pixels/frame
Overall movement direction: right

=== TEST RESULTS SUMMARY ===
 sequence_id  avg_velocity  max_velocity direction  detection_count
           0      1.493566      5.912396     right              389

Results saved to: D:/Project_P1/results/video_analysis\sequence_test_results.csv

=== ANALYZING FLIGHT PATTERNS ===
 sequence_id  pattern_type  avg_velocity  velocity_std  avg_acceleration  detection_count
           0 Steady flight       1.39215   

### MODEL FRONT-END APPLET (DEMO)

In [ ]:
import sys
import os
import cv2
from PyQt5.QtWidgets import (QApplication, QMainWindow, QWidget, QLabel, 
                             QPushButton, QFileDialog, QVBoxLayout, QHBoxLayout,
                             QSlider, QSpinBox, QComboBox, QGroupBox, QMessageBox)
from PyQt5.QtCore import Qt
from PyQt5.QtGui import QPixmap, QImage, QFont
from ultralytics import YOLO

class YOLOViewer(QMainWindow):
    def __init__(self):
        super().__init__()
        self.model = None
        self.image_files = []
        self.current_image_index = -1
        self.current_image = None
        self.current_results = None
        self.conf_threshold = 0.25
        self.initUI()
        
    def initUI(self):
        self.setWindowTitle('YOLO Object Detection Viewer')
        self.setGeometry(100, 100, 1200, 800)
        
        # Central widget
        central_widget = QWidget()
        self.setCentralWidget(central_widget)
        
        # Main layout
        main_layout = QHBoxLayout(central_widget)
        
        # Left panel for controls
        left_panel = QWidget()
        left_panel.setMaximumWidth(300)
        left_layout = QVBoxLayout(left_panel)
        
        # Model loading section
        model_group = QGroupBox("Model Configuration")
        model_layout = QVBoxLayout(model_group)
        
        self.model_btn = QPushButton("Load YOLO Model")
        self.model_btn.clicked.connect(self.load_model)
        model_layout.addWidget(self.model_btn)
        
        self.model_label = QLabel("No model loaded")
        model_layout.addWidget(self.model_label)
        
        # Image selection section
        image_group = QGroupBox("Image Selection")
        image_layout = QVBoxLayout(image_group)
        
        self.image_dir_btn = QPushButton("Select Image Directory")
        self.image_dir_btn.clicked.connect(self.select_image_directory)
        image_layout.addWidget(self.image_dir_btn)
        
        self.image_label = QLabel("No images loaded")
        image_layout.addWidget(self.image_label)
        
        self.prev_btn = QPushButton("Previous Image")
        self.prev_btn.clicked.connect(self.prev_image)
        image_layout.addWidget(self.prev_btn)
        
        self.next_btn = QPushButton("Next Image")
        self.next_btn.clicked.connect(self.next_image)
        image_layout.addWidget(self.next_btn)
        
        # Confidence threshold section
        conf_group = QGroupBox("Detection Settings")
        conf_layout = QVBoxLayout(conf_group)
        
        conf_control_layout = QHBoxLayout()
        conf_control_layout.addWidget(QLabel("Confidence Threshold:"))
        
        self.conf_slider = QSlider(Qt.Horizontal)
        self.conf_slider.setRange(0, 100)
        self.conf_slider.setValue(25)
        self.conf_slider.valueChanged.connect(self.conf_slider_changed)
        conf_control_layout.addWidget(self.conf_slider)
        
        self.conf_spinbox = QSpinBox()
        self.conf_spinbox.setRange(0, 100)
        self.conf_spinbox.setValue(25)
        self.conf_spinbox.valueChanged.connect(self.conf_spinbox_changed)
        conf_control_layout.addWidget(self.conf_spinbox)
        
        conf_layout.addLayout(conf_control_layout)
        
        self.detect_btn = QPushButton("Run Detection")
        self.detect_btn.clicked.connect(self.run_detection)
        conf_layout.addWidget(self.detect_btn)
        
        # Add groups to left layout
        left_layout.addWidget(model_group)
        left_layout.addWidget(image_group)
        left_layout.addWidget(conf_group)
        left_layout.addStretch()
        
        # Right panel for image display
        right_panel = QWidget()
        right_layout = QVBoxLayout(right_panel)
        
        self.image_display = QLabel()
        self.image_display.setAlignment(Qt.AlignCenter)
        self.image_display.setMinimumSize(640, 480)
        self.image_display.setText("No image loaded")
        self.image_display.setStyleSheet("border: 1px solid gray; background-color: #f0f0f0;")
        right_layout.addWidget(self.image_display)
        
        # Results info
        self.results_label = QLabel("Detection results will appear here")
        self.results_label.setWordWrap(True)
        right_layout.addWidget(self.results_label)
        
        # Add panels to main layout
        main_layout.addWidget(left_panel)
        main_layout.addWidget(right_panel)
        
        # Set initial state
        self.update_ui_state()
        
    def update_ui_state(self):
        model_loaded = self.model is not None
        images_loaded = len(self.image_files) > 0
        
        self.image_dir_btn.setEnabled(True)
        self.prev_btn.setEnabled(images_loaded and self.current_image_index > 0)
        self.next_btn.setEnabled(images_loaded and self.current_image_index < len(self.image_files) - 1)
        self.detect_btn.setEnabled(model_loaded and images_loaded)
        self.conf_slider.setEnabled(model_loaded and images_loaded)
        self.conf_spinbox.setEnabled(model_loaded and images_loaded)
        
    def load_model(self):
        file_path, _ = QFileDialog.getOpenFileName(
            self, "Select YOLO Model", "", "PyTorch Model (*.pt)"
        )
        
        if file_path:
            try:
                self.model = YOLO(file_path)
                self.model_label.setText(f"Model loaded: {os.path.basename(file_path)}")
                self.update_ui_state()
            except Exception as e:
                QMessageBox.critical(self, "Error", f"Failed to load model: {str(e)}")
                
    def select_image_directory(self):
        directory = QFileDialog.getExistingDirectory(self, "Select Image Directory")
        
        if directory:
            self.image_files = []
            for ext in ["*.jpg", "*.jpeg", "*.png", "*.bmp"]:
                self.image_files.extend([os.path.join(directory, f) for f in os.listdir(directory) 
                                       if f.lower().endswith(ext.replace("*", ""))])
            
            if self.image_files:
                self.current_image_index = 0
                self.load_current_image()
                self.image_label.setText(f"Loaded {len(self.image_files)} images")
            else:
                self.image_label.setText("No images found in directory")
                self.current_image_index = -1
                
            self.update_ui_state()
            
    def load_current_image(self):
        if 0 <= self.current_image_index < len(self.image_files):
            image_path = self.image_files[self.current_image_index]
            self.current_image = cv2.imread(image_path)
            if self.current_image is not None:
                self.display_image(self.current_image)
                self.results_label.setText(f"Loaded: {os.path.basename(image_path)}")
            else:
                self.results_label.setText(f"Failed to load image: {os.path.basename(image_path)}")
                
    def display_image(self, image):
        # Convert BGR to RGB
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w, ch = image_rgb.shape
        bytes_per_line = ch * w
        qt_image = QImage(image_rgb.data, w, h, bytes_per_line, QImage.Format_RGB888)
        pixmap = QPixmap.fromImage(qt_image)
        
        # Scale pixmap to fit label while maintaining aspect ratio
        self.image_display.setPixmap(
            pixmap.scaled(
                self.image_display.width(),
                self.image_display.height(),
                Qt.KeepAspectRatio,
                Qt.SmoothTransformation
            )
        )
        
    def prev_image(self):
        if self.current_image_index > 0:
            self.current_image_index -= 1
            self.load_current_image()
            self.update_ui_state()
            
    def next_image(self):
        if self.current_image_index < len(self.image_files) - 1:
            self.current_image_index += 1
            self.load_current_image()
            self.update_ui_state()
            
    def conf_slider_changed(self, value):
        self.conf_threshold = value / 100.0
        self.conf_spinbox.setValue(value)
        
    def conf_spinbox_changed(self, value):
        self.conf_threshold = value / 100.0
        self.conf_slider.setValue(value)
        
    def run_detection(self):
        if self.model is None or self.current_image is None:
            return
            
        try:
            # Run detection
            results = self.model(self.current_image, conf=self.conf_threshold, imgsz=640)
            self.current_results = results[0]
            
            # Plot results on image
            result_image = self.current_results.plot()
            
            # Display the result
            self.display_image(result_image)
            
            # Display detection info
            if self.current_results.boxes is not None:
                num_detections = len(self.current_results.boxes)
                classes = self.current_results.boxes.cls
                confs = self.current_results.boxes.conf
                
                class_counts = {}
                for i in range(num_detections):
                    class_id = int(classes[i])
                    class_name = self.current_results.names[class_id]
                    conf = float(confs[i])
                    
                    if class_name not in class_counts:
                        class_counts[class_name] = 0
                    class_counts[class_name] += 1
                
                info_text = f"Detected {num_detections} objects:\n"
                for class_name, count in class_counts.items():
                    info_text += f"- {class_name}: {count}\n"
                    
                self.results_label.setText(info_text)
            else:
                self.results_label.setText("No objects detected")
                
        except Exception as e:
            QMessageBox.critical(self, "Error", f"Detection failed: {str(e)}")
            
def main():
    app = QApplication(sys.argv)
    app.setStyle('Fusion')
    
    # Set application font
    font = QFont()
    font.setPointSize(10)
    app.setFont(font)
    
    viewer = YOLOViewer()
    viewer.show()
    
    sys.exit(app.exec_())

if __name__ == '__main__':
    main()